# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [4]:
# Load environment variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Initialize LLM client
client = LLM(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY
)

# Load ChromaDB collection from Part 1
chroma_client = chromadb.PersistentClient(path="./chroma_db")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"
)
collection = chroma_client.get_or_create_collection(
    name="games",
    embedding_function=embedding_fn
)

print(f"✓ ChromaDB collection loaded: {collection.count()} games available")

✓ ChromaDB collection loaded: 15 games available


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
from lib.tooling import tool

@tool(name="retrieve_game", description="Semantic search: Finds game results in the vector DB using ChromaDB. Returns the most relevant games matching the query with platform, name, year, and confidence score.")
def retrieve_game(query: str) -> dict:
    """
    Retrieves games from ChromaDB based on semantic similarity to the query.
    
    Args:
        query: A question about games or game-related topics
        
    Returns:
        dict with:
        - results: list of game documents with metadata
        - confidence: semantic search confidence score (1 - distance)
        - distance: raw distance metric from ChromaDB
    """
    # Retrieve from ChromaDB
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    
    # Format results
    retrieved_games = []
    if results['documents'] and len(results['documents']) > 0:
        for i, doc in enumerate(results['documents'][0]):
            distance = results['distances'][0][i] if results['distances'] else 0
            confidence = 1 - distance  # Convert distance to confidence
            
            metadata = results['metadatas'][0][i] if results['metadatas'] else {}
            
            retrieved_games.append({
                'document': doc,
                'platform': metadata.get('platform', 'Unknown'),
                'name': metadata.get('name', 'Unknown'),
                'year': metadata.get('year', 'Unknown'),
                'confidence': round(confidence, 2)
            })
    
    return {
        'results': retrieved_games,
        'query': query,
        'num_results': len(retrieved_games)
    }

#### Evaluate Retrieval Tool

In [6]:
from pydantic import BaseModel

class EvaluationReport(BaseModel):
    """Evaluation report for retrieved documents"""
    is_relevant: bool
    confidence: float
    reasoning: str

@tool(name="evaluate_retrieval", description="Evaluates if retrieved documents are relevant and sufficient to answer the user's question. Returns relevance judgment with confidence score and reasoning.")
def evaluate_retrieval(query: str, retrieval_result: dict) -> dict:
    """
    Uses an LLM to evaluate if retrieved documents are sufficient to answer the query.
    
    Args:
        query: The original user question
        retrieval_result: Results from retrieve_game tool
        
    Returns:
        dict with evaluation report including relevance, confidence, and reasoning
    """
    # Format documents for evaluation
    documents_text = "\n".join([
        f"- {r['name']} ({r['platform']}, {r['year']}): {r['document']}"
        for r in retrieval_result.get('results', [])
    ])
    
    # Create evaluation prompt
    eval_prompt = f"""Your task is to evaluate if the retrieved documents are sufficient to answer the user's question.

User Question: {query}

Retrieved Documents:
{documents_text}

Analyze whether these documents contain enough information to answer the question. Consider:
1. Do the documents directly address the question?
2. Is the information specific enough?
3. What is your confidence level (0.0 to 1.0)?

Provide a JSON response with:
- "is_relevant": boolean indicating if documents are useful
- "confidence": confidence score (0.0 to 1.0)
- "reasoning": explanation of your assessment
"""
    
    # Use LLM to evaluate
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": eval_prompt}],
        temperature=0.3
    )
    
    eval_text = response.choices[0].message.content
    
    # Try to parse JSON response
    try:
        import json
        # Extract JSON from response
        json_start = eval_text.find('{')
        json_end = eval_text.rfind('}') + 1
        if json_start != -1 and json_end > json_start:
            json_str = eval_text[json_start:json_end]
            eval_data = json.loads(json_str)
            report = EvaluationReport(
                is_relevant=eval_data.get('is_relevant', True),
                confidence=float(eval_data.get('confidence', 0.5)),
                reasoning=eval_data.get('reasoning', eval_text)
            )
        else:
            # Default if parsing fails
            report = EvaluationReport(
                is_relevant=True,
                confidence=0.7,
                reasoning=eval_text
            )
    except Exception as e:
        report = EvaluationReport(
            is_relevant=True,
            confidence=0.6,
            reasoning=f"Evaluation: {eval_text}"
        )
    
    return {
        'is_relevant': report.is_relevant,
        'confidence': report.confidence,
        'reasoning': report.reasoning
    }

#### Game Web Search Tool

In [7]:
from tavily import TavilyClient

@tool(name="game_web_search", description="Performs web search using Tavily API to find current information about games when internal knowledge is insufficient. Returns search results with answers and sources.")
def game_web_search(query: str) -> dict:
    """
    Searches the web for game-related information using Tavily API.
    
    Args:
        query: Search query about games
        
    Returns:
        dict with:
        - answer: synthesized answer from web search results
        - sources: list of source URLs
        - confidence: confidence in the answer (0.5-1.0 range)
    """
    try:
        # Initialize Tavily client
        tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        
        # Perform web search
        search_response = tavily_client.search(query, max_results=5)
        
        # Extract answer and sources
        answer = search_response.get("answer", "No answer found")
        sources = [result.get("url", "") for result in search_response.get("results", [])]
        
        return {
            'answer': answer,
            'sources': sources,
            'confidence': 0.8,
            'query': query
        }
    except Exception as e:
        return {
            'answer': f"Web search failed: {str(e)}",
            'sources': [],
            'confidence': 0.0,
            'query': query,
            'error': str(e)
        }

### Agent

In [8]:
# Create Agent with system instructions and all tools
from lib.agents import Agent

# System prompt for the agent
system_prompt = """You are an expert gaming AI assistant specializing in video game information and history.

Your primary role is to:
1. Answer questions about video games using the game database (retrieve_game tool)
2. Evaluate if retrieved information is sufficient (evaluate_retrieval tool)
3. Fall back to web search for current or missing information (game_web_search tool)

Guidelines:
- First, try to retrieve information from the internal game database
- Evaluate the relevance of retrieved results
- If results are not relevant (confidence < 0.5) or insufficient, use web search
- Always provide sources and confidence levels in your responses
- Be accurate and cite game platforms, years, and publishers when available
- If you're unsure about information, acknowledge the uncertainty

When answering:
- Use retrieved data as primary source
- Cross-reference with web search for verification
- Provide complete answers with platform, release year, and relevant details
- Explain your reasoning for tool usage"""

# Create the agent
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=system_prompt,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3
)

print("✓ Agent created successfully with 3 tools: retrieve_game, evaluate_retrieval, game_web_search")

✓ Agent created successfully with 3 tools: retrieve_game, evaluate_retrieval, game_web_search


In [9]:
# Test queries with the agent
import json

test_queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?"
]

print("=" * 70)
print("AGENT TEST QUERIES")
print("=" * 70)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}] {query}")
    print("-" * 70)
    
    try:
        # Invoke agent with session ID for state tracking
        response = agent.invoke(query, session_id=f"test_session_{i}")
        
        # Display response
        print(f"Answer: {response}")
        print()
    except Exception as e:
        print(f"Error: {str(e)}")
        print()

print("=" * 70)
print("✓ Agent testing complete")
print("=" * 70)

AGENT TEST QUERIES

[Query 1] When was Pokémon Gold and Silver released?
----------------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Error: 'str' object has no attribute 'get'


[Query 2] Which one was the first 3D platformer Mario game?
----------------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Error: 'str' object has no attribute 'get'


[Query 3] Was Mortal Kombat X released for Playstation 5?
----------------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes